# Swept solid beam — v2: `sweep` helper + `node_to_surface_spring`

Same problem as `example_LTB_swept_solid.ipynb` — imperfect 3-D
solid I/strip beam with geometric imperfection baked into the
volume — but this time we use **two new apeGmsh primitives** that
together eliminate the manual gmsh.occ glue and the per-face BC
distribution from v1:

### `m.model.geometry.sweep(profile, path, …)`

Wraps `gmsh.model.occ.addWire` + `gmsh.model.occ.addPipe` and
**handles the two cleanup steps that v1 had to do by hand**:

1. Removes the original profile face (so the pipe's own start cap is
   the unique surface at the path start — v1 picked up both and
   their mesh nodes doubled).
2. Removes the path curves, which would otherwise sit along the
   solid centroid line and mesh into stray `line2` elements whose
   nodes are not shared with any tet4 (floating null-space DOFs).

It returns a dict `{'volume', 'start_cap', 'end_cap'}` with the
tags already identified, so labelling the end caps as physical
groups is a one-liner.

### `m.constraints.node_to_surface_spring(master, slave)`

Spring-based variant of `node_to_surface`. Same phantom-node
topology, but the master → phantom link is downstream-tagged for
emission as a **stiff `elasticBeamColumn` element** instead of a
kinematic `rigidLink('beam', …)` constraint. The difference matters
when the master has free rotation DOFs that receive moment
loading, because the element-based stiffness contributes directly
to the master's rotation diagonal rather than relying on constraint
back-propagation through `ndf=3` tet4 face nodes.

See `Constraints.NodeToSurfaceSpringDef` for the full rationale.

### What this notebook does

1. Build a path with `add_line` → `replace_line` (sine imperfection)
2. Build a rectangular profile with `gmsh.model.occ`
3. **`m.model.geometry.sweep`** to produce the solid + get the cap
   tags back
4. Add `ref_pin` and `ref_roller` 0-D points and couple each to its
   end cap via **`m.constraints.node_to_surface_spring`**
5. Mesh
6. Emit to OpenSees — the coupling chain is now
   `stiff_beam_groups()` + `equal_dofs()`, not
   `rigid_link_groups()` + `equal_dofs()`
7. Apply fork-support BCs and a pure end moment **at the ref
   points**, not distributed across every face node
8. Run a linear elastic analysis and visualise

In [ ]:
from apeGmsh import apeGmsh, Results
import gmsh
import numpy as np
import matplotlib.pyplot as plt

# ---- Steel material ----------------------------------------------
E   = 200_000.0
nu  = 0.3
G   = E / (2.0 * (1 + nu))

# ---- Rectangular strip section -----------------------------------
b_depth = 200.0
t_thk   = 20.0

A        = b_depth * t_thk
I_strong = t_thk * b_depth ** 3 / 12.0
I_weak   = b_depth * t_thk ** 3 / 12.0
J        = (1.0 / 3.0) * b_depth * t_thk ** 3 * (1 - 0.63 * t_thk / b_depth)

# ---- Beam path + imperfection ------------------------------------
L       = 4000.0
delta_0 = L / 1000.0
lc      = 30.0

# ---- Stiff-beam section properties for the spring links --------
# These are the "effective rigid link" section properties used by
# the elasticBeamColumn elements that replace the rigidLink at the
# end-face couplings. They should be orders of magnitude stiffer
# than the beam material so the links behave as approximately
# rigid. The specific numbers are not critical — anything 4+
# orders of magnitude larger than the natural section values works.
A_stiff = 1.0e6
I_stiff = 1.0e10
J_stiff = 1.0e10

M_cr_line = (np.pi / L) * np.sqrt(E * I_weak * G * J)

print(f'Rectangle {b_depth:.0f} x {t_thk:.0f} mm  (I_strong/I_weak = {I_strong/I_weak:.0f})')
print(f'L = {L:.0f} mm    delta_0 = {delta_0:.2f} mm')
print(f'Line-element M_cr (no warping) = {M_cr_line/1e6:.2f} kN*m')

## 1. Build the imperfect path and sweep it into a solid

The whole sweep + cleanup + cap identification is now a single
`m.model.geometry.sweep(prof, path)` call. It returns a dict whose
keys are the new volume tag and the two end-cap surface tags.
Labelling them as physical groups is three one-liners.

In [ ]:
m = apeGmsh(model_name='swept_v2', verbose=False)
m.begin()

# --- Imperfect path
p0 = m.model.geometry.add_point(0, 0, 0, lc=lc)
p1 = m.model.geometry.add_point(L, 0, 0, lc=lc)
path_line = m.model.geometry.add_line(p0, p1)
path_tags = m.model.geometry.replace_line(
    path_line,
    magnitude=delta_0,
    direction=(0, 1, 0),
    shape='sine',
    n_segments=16,
)

# --- Rectangular profile at x = 0
corners = [
    (0, -t_thk/2, -b_depth/2),
    (0,  t_thk/2, -b_depth/2),
    (0,  t_thk/2,  b_depth/2),
    (0, -t_thk/2,  b_depth/2),
]
cp = [gmsh.model.occ.addPoint(*c) for c in corners]
cl = [gmsh.model.occ.addLine(cp[i], cp[(i + 1) % 4]) for i in range(4)]
prof_loop = gmsh.model.occ.addCurveLoop(cl)
prof_face = gmsh.model.occ.addPlaneSurface([prof_loop])
gmsh.model.occ.synchronize()

# --- Sweep (the new helper — wraps addWire, addPipe, and cleanup)
swept = m.model.geometry.sweep(prof_face, path_tags, label='beam_body')
print(f'sweep returned: {swept}')

# --- PG labels for volume + caps
gmsh.model.addPhysicalGroup(
    3, [swept['volume']], tag=-1, name='pg_beam')
gmsh.model.addPhysicalGroup(
    2, [swept['start_cap']], tag=-1, name='pin_face')
gmsh.model.addPhysicalGroup(
    2, [swept['end_cap']], tag=-1, name='roller_face')
gmsh.model.occ.synchronize()

## 2. Add the two reference points and couple them via `node_to_surface_spring`

Each reference point is a 0-D labelled geometry entity co-located
with the corresponding end face's centroid. The coupling is the
**spring** variant — downstream the master → phantom link becomes
a stiff `elasticBeamColumn` element instead of a `rigidLink`
constraint, so the master rotation DOFs receive element stiffness
directly and the system stays well-conditioned under moment
loading.

In [ ]:
# Offset the reference points slightly OUTSIDE the corresponding end
# face (by a small gap in the beam-axis direction). This avoids the
# degenerate case where a face mesh node happens to land on exactly
# (0, 0, 0) — if ref_pin were at (0, 0, 0) the stiff-beam link to
# that phantom would be zero length and the coordinate transform
# would refuse to initialise. A 50 mm offset is well inside the
# stiff-beam regime (section stiffness ≫ flexural compliance over
# 50 mm) so the effective BC is still "pinned at the face".
REF_OFFSET = 50.0

m.model.geometry.add_point(-REF_OFFSET,       0, 0, lc=lc, label='ref_pin')
m.model.geometry.add_point(L + REF_OFFSET,    0, 0, lc=lc, label='ref_roller')

m.constraints.node_to_surface_spring('ref_pin',    'pin_face')
m.constraints.node_to_surface_spring('ref_roller', 'roller_face')

print(f'constraint defs: '
      f'{[type(d).__name__ for d in m.constraints.constraint_defs]}')

## 3. Mesh and inspect

In [ ]:
m.mesh.sizing.set_global_size(lc)
m.mesh.generation.generate(dim=3)

fem = m.mesh.queries.get_fem_data(remove_orphans=True)
m.end()

print(f'total mesh nodes: {len(fem.nodes.ids)}')
for g in fem.elements:
    print(f'  {g.type_name:6s} n={len(g)}')
print()
print(f'pg_beam     nodes: {len(fem.nodes.get_ids(target="pg_beam"))}')
print(f'pin_face    nodes: {len(fem.nodes.get_ids(target="pin_face"))}')
print(f'roller_face nodes: {len(fem.nodes.get_ids(target="roller_face"))}')
print(f'ref_pin     : {list(int(n) for n in fem.nodes.get_ids(target="ref_pin"))}')
print(f'ref_roller  : {list(int(n) for n in fem.nodes.get_ids(target="ref_roller"))}')
print()
print(f'compound records: {len(fem.nodes.constraints)}')
for rec in fem.nodes.constraints:
    print(f'  kind={rec.kind!r} phantoms={len(rec.phantom_nodes)}')
n_stiff = sum(len(s) for _, s in fem.nodes.constraints.stiff_beam_groups())
n_ed    = sum(1 for _ in fem.nodes.constraints.equal_dofs())
print(f'stiff beam pairs: {n_stiff}')
print(f'equalDOF pairs  : {n_ed}')

## 4. Emit the OpenSees model

`ndm=3, ndf=6` — the ref points and phantoms carry all six DOFs so
the stiff beam elements can operate on them. Solid tet nodes are
`-ndf 3`-overridden.

The emission differs from the normal `node_to_surface` flow in
exactly one place: instead of iterating `rigid_link_groups()` and
calling `ops.rigidLink('beam', …)`, we iterate
**`stiff_beam_groups()`** and emit `ops.element('elasticBeamColumn', …)`
with the stiff section properties.

In [ ]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)
ops.timeSeries('Linear', 1)

# --- Solid tet nodes (ndf=3) -------------------------------------
n_solid = 0
for nid, xyz in fem.nodes.get(target='pg_beam'):
    ops.node(int(nid), *xyz, '-ndf', 3)
    n_solid += 1

# --- Ref points (ndf=6) ------------------------------------------
for nid, xyz in fem.nodes.get(target=['ref_pin', 'ref_roller']):
    ops.node(int(nid), *xyz)

# --- Phantom nodes (ndf=6) ---------------------------------------
n_phantom = 0
for nid, xyz in fem.nodes.constraints.phantom_nodes():
    ops.node(int(nid), *xyz)
    n_phantom += 1

# --- Solid material + tet4 elements ------------------------------
ops.nDMaterial('ElasticIsotropic', 1, E, nu)
n_tet = 0
max_eid = 0
for group in fem.elements.get(element_type='tet4'):
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', int(eid), *conn, 1)
        max_eid = max(max_eid, int(eid))
        n_tet += 1

# --- Stiff beam elements for the node_to_surface_spring chains --
# One elasticBeamColumn per phantom, between the ref point and the
# phantom. Section properties A_stiff / I_stiff / J_stiff are chosen
# 4+ orders of magnitude larger than the solid's natural values so
# that the beam acts as a near-rigid kinematic bridge. With the
# reference point offset 50 mm outside each end face, every beam
# has a nonzero length and the directions fan out in the half-space
# pointing away from the face.
#
# Coordinate-transformation vec_xz choice: a non-axis vector
# (1, 1, 1)/sqrt(3) is guaranteed to be non-parallel to every beam
# axis unless the beam runs along the (1, 1, 1) direction, which
# is impossible for stiff beams spanning (-50, 0, 0) → (0, y, z)
# and (L, 0, 0) → (L+50 shifted mirror).
ops.geomTransf(
    'Linear', 1,
    1.0 / np.sqrt(3),
    1.0 / np.sqrt(3),
    1.0 / np.sqrt(3),
)

next_eid = max_eid + 1
n_stiff = 0
for master, slaves in fem.nodes.constraints.stiff_beam_groups():
    for slave in slaves:
        ops.element(
            'elasticBeamColumn', next_eid,
            int(master), int(slave),
            A_stiff, E, G, J_stiff, I_stiff, I_stiff, 1,
        )
        next_eid += 1
        n_stiff += 1

# --- EqualDOF: phantom -> solid face node (translations only) ----
n_ed = 0
for pair in fem.nodes.constraints.equal_dofs():
    ops.equalDOF(pair.master_node, pair.slave_node, *pair.dofs)
    n_ed += 1

print(f'solid nodes  (ndf=3) : {n_solid}')
print(f'phantom nodes (ndf=6): {n_phantom}')
print(f'tet4 elements        : {n_tet}')
print(f'stiff beam elements  : {n_stiff}')
print(f'equalDOF pairs       : {n_ed}')

## 5. Apply BCs at the ref points and load

With the spring coupling in place, all BCs and loads go through the
**reference points only**. The solid end faces are driven via the
stiff-beam + equalDOF chain.

* `ref_pin`    — `ux = uy = uz = 0`, `rx = 0`, `ry`, `rz` free
* `ref_roller` — `uy = uz = 0`, `rx = 0`, `ry`, `rz` free, `ux` also fixed
  (axial release left disabled for solver robustness — the beam is
  free to shorten elastically through the solid)
* Load: `+My` at `ref_pin`, `-My` at `ref_roller` → uniform internal
  `My` along the span

In [ ]:
ref_pin_id  = int(fem.nodes.get_ids(target='ref_pin')[0])
ref_roll_id = int(fem.nodes.get_ids(target='ref_roller')[0])

ops.fix(ref_pin_id,  1, 1, 1, 1, 0, 0)
ops.fix(ref_roll_id, 1, 1, 1, 1, 0, 0)

ops.pattern('Plain', 1, 1)
ops.load(ref_pin_id,  0, 0, 0,  0, +1.0, 0)
ops.load(ref_roll_id, 0, 0, 0,  0, -1.0, 0)

ops.constraints('Transformation')
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-7, 200)
ops.algorithm('Linear')

M_target = 0.6 * M_cr_line
n_steps  = 20
dM       = M_target / n_steps
ops.integrator('LoadControl', dM)
ops.analysis('Static')

beam_ids    = np.array(
    [int(n) for n in fem.nodes.get_ids(target='pg_beam')])
beam_coords = fem.nodes.get_coords(target='pg_beam')
d_mid = np.linalg.norm(
    beam_coords - np.array([L/2, 0, 0]), axis=1)
mid_nid = int(beam_ids[int(np.argmin(d_mid))])
print(f'midspan probe node {mid_nid} at '
      f'{tuple(beam_coords[int(np.argmin(d_mid))].round(1))}')

emitted_ids = set(int(n) for n in beam_ids)
# Also need the ref points + phantoms for the viewer
emitted_ids.add(ref_pin_id)
emitted_ids.add(ref_roll_id)
for pid in fem.nodes.constraints.phantom_nodes().ids:
    emitted_ids.add(int(pid))
id_to_row = {int(nid): i for i, nid in enumerate(fem.nodes.ids)}

hist_M    = []
hist_Ry   = []
hist_midy = []
hist_midz = []
disp_per_step: list[np.ndarray] = []
n_nodes = len(fem.nodes.ids)

for step in range(n_steps):
    ok = ops.analyze(1)
    if ok != 0:
        print(f'failed at step {step + 1}, ok={ok}')
        break
    M_now = (step + 1) * dM
    d_ref = ops.nodeDisp(ref_pin_id)
    d_mid = ops.nodeDisp(mid_nid)
    hist_M.append(M_now)
    hist_Ry.append(d_ref[4])
    hist_midy.append(d_mid[1])
    hist_midz.append(d_mid[2])

    d_full = np.zeros((n_nodes, 3), dtype=np.float64)
    for nid in emitted_ids:
        if nid not in id_to_row:
            continue
        di = ops.nodeDisp(int(nid))
        row = id_to_row[nid]
        d_full[row, 0] = di[0]
        d_full[row, 1] = di[1]
        d_full[row, 2] = di[2]
    disp_per_step.append(d_full)

if not hist_M:
    raise RuntimeError('analysis did not converge on any step')

print(f'converged in {len(hist_M)} of {n_steps} steps')
print(f'final M           = {hist_M[-1]/1e6:.3f} kN*m  '
      f'({hist_M[-1]/M_cr_line*100:.1f}% M_cr_line)')
print(f'final ref_pin Ry  = {hist_Ry[-1]:+.5e} rad')
print(f'final midspan uy  = {hist_midy[-1]:+.5f} mm')
print(f'final midspan uz  = {hist_midz[-1]:+.5f} mm')

## 6. M vs midspan + M vs ref-pin rotation

The `ref_pin.ry` rotation is only physically meaningful with the
spring coupling — before, the constraint-based chain was what made
its reduced-system conditioning fragile. Now it has a direct
stiffness contribution from the stiff `elasticBeamColumn` elements
and behaves as a normal free DOF.

In [ ]:
hist_M    = np.asarray(hist_M)
hist_Ry   = np.asarray(hist_Ry)
hist_midy = np.asarray(hist_midy)
hist_midz = np.asarray(hist_midz)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

ax = axes[0]
ax.plot(hist_midz, hist_M / 1e6, 'o-', lw=1.4, ms=3, color='tab:blue')
ax.set_xlabel('midspan uz [mm]')
ax.set_ylabel('M  [kN*m]')
ax.set_title('Strong-axis bending')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(hist_midy, hist_M / 1e6, 'o-', lw=1.4, ms=3, color='tab:red')
ax.axvline(delta_0, color='tab:gray', lw=0.8, ls=':',
           label=f'\u03b4\u2080 = {delta_0:.2f} mm')
ax.set_xlabel('midspan uy [mm]')
ax.set_ylabel('M  [kN*m]')
ax.set_title('Weak-axis (imperfection)')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right')

ax = axes[2]
ax.plot(np.degrees(hist_Ry), hist_M / 1e6, 'o-', lw=1.4, ms=3,
        color='tab:purple')
ax.set_xlabel('ref_pin Ry [deg]')
ax.set_ylabel('M  [kN*m]')
ax.set_title('Reference rotation')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Time-series viewer

In [ ]:
steps = []
for M_i, d_step in zip(hist_M, disp_per_step):
    u_mag = np.linalg.norm(d_step, axis=1)
    steps.append({
        'time': float(M_i),
        'point_data': {
            'Displacement': d_step,
            '|U|':          u_mag,
            'Uy':           d_step[:, 1],
            'Uz':           d_step[:, 2],
        },
    })

print(f'time-series steps: {len(steps)}')
print(f'time range       : {steps[0]["time"]/1e6:.2f} to '
      f'{steps[-1]["time"]/1e6:.2f} kN*m')

results = Results.from_fem(
    fem,
    steps=steps,
    include_pgs=['pg_beam'],
    name='swept_solid_v2',
)
results.viewer(blocking=False)